In [ ]:
"Download nltk models - run only once"

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

In [3]:
"Convert to wav"

import subprocess

def convert_mp3_to_wav(mp3_file, wav_file):
    command = [
        "ffmpeg",
        "-i", mp3_file,  # Input file
        wav_file         # Output file
    ]
    subprocess.run(command, check=True)
    print(f"Converted {mp3_file} to {wav_file}")

# Example usage
convert_mp3_to_wav("en_2.mp3", "en_2.wav")

from IPython.display import Audio, display

audio_sample_1 = Audio("en_2.wav", autoplay=False)
display(audio_sample_1)

Converted en_2.mp3 to en_2.wav


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [1]:
"Align transcript with original text"

import difflib
import re
from typing import List, Tuple, Optional


def _normalize_word(word: str) -> str:
    """
    Normalize a single word by removing punctuation and converting to lowercase.
    
    :param word: Input word
    :return: Normalized word
    """
    # Remove surrounding punctuation, but keep internal punctuation for special cases like contractions
    return re.sub(r"^[^\w\'-]+|[^\w\'-]+$", '', word).lower()


def _split_text_to_words(text: str) -> List[str]:
    """
    Split text into words while preserving original form.
    
    :param text: Original text
    :return: List of words
    """
    # Use regex to split words while keeping punctuation attached
    return re.findall(r"[\w'-]+[.,!?']*", text)


def _find_best_match(
        transcript_word: str, 
        remaining_text_words: List[str]
    ) -> Optional[Tuple[str, int]]:
        """
        Find the best matching word in text.
        
        :param transcript_word: Word from transcript
        :param remaining_text_words: Remaining words in original text
        :return: Best matched word and its index, or None
        """
        # Normalize transcript word for comparison
        norm_transcript_word = _normalize_word(transcript_word)
        
        # First, try exact match
        for i, text_word in enumerate(remaining_text_words):
            norm_text_word = _normalize_word(text_word)
            if norm_transcript_word == norm_text_word:
                return text_word, i
        
        # If no exact match, try similarity matching
        best_match = None
        best_similarity = 0
        
        for i, text_word in enumerate(remaining_text_words):
            norm_text_word = _normalize_word(text_word)
            
            # Compute similarity
            similarity = difflib.SequenceMatcher(
                None, 
                norm_transcript_word, 
                norm_text_word
            ).ratio()
            
            # Update best match if similarity is higher
            if similarity > best_similarity and similarity >= 0.7:
                best_match = (text_word, i)
                best_similarity = similarity
        
        return best_match


def align_transcript(text: str, transcript: List[List[float]]) -> List[List[float]]:
    """
    Align transcript words with original text, correcting words while preserving timing.
    
    :return: Aligned transcript
    """
    # Normalize text 
    original_text = text
    
    # Compute text words for efficient matching
    text_words = _split_text_to_words(text)
    
    aligned_transcript = []
    text_word_index = 0
    
    for transcript_entry in transcript:
        transcript_word = transcript_entry[0].strip()
        start_time, end_time = transcript_entry[1], transcript_entry[2]
        
        # If we've exhausted text words, break
        if text_word_index >= len(text_words):
            aligned_transcript.append(transcript_entry)
            continue
        
        # Try to find the best match
        best_match = _find_best_match(
            transcript_word, 
            text_words[text_word_index:]
        )
        
        if best_match is not None:
            matched_word, offset = best_match
            text_word_index += offset + 1
            # Create new entry with the original text word but keep original timing
            aligned_transcript.append([matched_word, start_time, end_time])
        else:
            # Fallback: use original text word
            aligned_transcript.append([text_words[text_word_index], start_time, end_time])
            text_word_index += 1
    
    return aligned_transcript

In [2]:
import torch
import time
import json
import numpy as np
from scipy.io.wavfile import write
from nltk.tokenize import sent_tokenize
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
from IPython.display import Audio, display
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
import gc
import io
import ffmpeg

def timeit_decorator(func):
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"Execution time of {func.__name__}: {end_time - start_time} seconds")
        return result
    return wrapper

CUDA_DEVICE = 1
# TTS
tts_config = XttsConfig()
tts_config.load_json("XTTS-v2/config.json")
tts_model = Xtts.init_from_config(tts_config)
tts_model.load_checkpoint(tts_config, checkpoint_dir="XTTS-v2", eval=True)
tts_model.cuda(CUDA_DEVICE)

reference_audios_en = [
    "./storytime-voice-sample-en-1.wav",
    "./storytime-voice-sample-en-2.wav",
    "./storytime-voice-sample-en-3.wav",
]



en_gpt_cond_latent, en_speaker_embedding = tts_model.get_conditioning_latents(
    audio_path=reference_audios_en
)

supported_languages = ["en"]
PAUSE_DURATION = 0.5
SAMLE_RATE = 24000
PAUSE = np.zeros(int(SAMLE_RATE * PAUSE_DURATION))

tts_config = XttsConfig()
tts_config.load_json("XTTS-v2/config.json")
tts_model = Xtts.init_from_config(tts_config)
tts_model.load_checkpoint(tts_config, checkpoint_dir="XTTS-v2", eval=True)
tts_model.cuda(CUDA_DEVICE)

# reference_audios_en = [
#     "./storytime-voice-sample-en-1.wav",
#     "./storytime-voice-sample-en-2.wav",
#     "./storytime-voice-sample-en-3.wav",
# ]

reference_audios_en = ["olia_voice_sample.wav"]

en_gpt_cond_latent, en_speaker_embedding = tts_model.get_conditioning_latents(
    audio_path=reference_audios_en
)

supported_languages = ["en"]
PAUSE_DURATION = 0.5
SAMLE_RATE = 24000
PAUSE = np.zeros(int(SAMLE_RATE * PAUSE_DURATION))


# Transcript
device = f"cuda:{CUDA_DEVICE}"
torch_dtype = torch.float16

TRANSCRIPT_MODEL_ID = "./whisper-large-v3-turbo"
transcript_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    TRANSCRIPT_MODEL_ID,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
)
transcript_model.to(device)
transcript_processor = AutoProcessor.from_pretrained(TRANSCRIPT_MODEL_ID)

transcript_pipe = pipeline(
    "automatic-speech-recognition",
    model=transcript_model,
    tokenizer=transcript_processor.tokenizer,
    feature_extractor=transcript_processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)


def generate(prompt, language):
    """Read loud text in language"""
    if language not in supported_languages:
        raise UnsupportedLanguageException()

    combined_audio = []

    sentences = sent_tokenize(prompt)
    for sentence in sentences:
        outputs = tts_model.inference(
            text=sentence,
            gpt_cond_latent=en_gpt_cond_latent,
            speaker_embedding=en_speaker_embedding,
            language=language,
            enable_text_splitting=True,
        )
        combined_audio.append(outputs["wav"])
        combined_audio.append(PAUSE)
    combined_audio.pop()
    audio = np.concatenate(combined_audio)
    return audio


def encode_to_wav(audio_array):
    """Encodes a NumPy array to WAV in memory."""
    audio_bytes = audio_array.astype(np.float32).tobytes()

    # Use FFmpeg to encode WAV
    process = (
        ffmpeg
        .input('pipe:0', format='f32le', ac=1, ar=SAMLE_RATE)
        .output('pipe:1', format='wav')
        .run_async(pipe_stdin=True, pipe_stdout=True, pipe_stderr=True)
    )
    wav_data, _ = process.communicate(input=audio_bytes)
    return wav_data


def transcribe(audio_data):
    """Transcribes the voice with timestamps"""
    audio_input = {"raw": audio_data, "sampling_rate": SAMLE_RATE}
    output = transcript_pipe(audio_input, return_timestamps="word")
    return [
        [
            chunk["text"],
            chunk["timestamp"][0],
            chunk["timestamp"][1],
        ]
        for chunk in output["chunks"]
    ]


def voice_pipe(text: str, language: str) -> tuple[bytes, str]:
    """
    Generates audio with voice and transcription with timestamps

    Args:
        text (str): Text to read
        language (str): Language code of the text

    Returns:
        tuple[bytes, bytes]: wav_bytes, notation_bytes
    """
    audio_data = generate(text, language)
    notation = transcribe(audio_data)
    aligned_notation = align_transcript(text, notation)
    notation = json.dumps(aligned_notation)
    wav_bytes = encode_to_wav(audio_data)
    return wav_bytes, notation


In [3]:
text = "Once upon a time, there was a magical forest full of wondrous sights and sounds. The trees were tall and mighty, their leaves rustling in the gentle breeze. Colorful flowers bloomed everywhere, and the birds sang sweet melodies, welcoming all who entered. In this enchanting forest lived a little fox named Fanny. Fanny was unlike any other fox. She was curious and loved to explore every nook and cranny of her home. One bright sunny morning, while Fanny was playfully darting between the trees, she heard a soft chirping sound that caught her attention. 'What could that be?' she wondered. Without a second thought, Fanny followed the sound, eager to see who needed help."
text = "I dont like to play with Monster High dolls. And I will give half of the Ore I get in Valheim to my best brother in the world." 
audio_data, text_transcribe = voice_pipe(text, 'en')
audio = Audio(data=audio_data)
display(audio)

def write_to_json(data):
    with open('output.json', 'w', encoding='utf-8') as f:
        f.write(data)

def write_to_wav(byte_data):
    with open('output.wav', 'wb') as f:
        f.write(byte_data)

write_to_json(text_transcribe)
write_to_wav(audio_data)
text_transcribe

/home/pavlo/miniconda3/lib/python3.12/site-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
WhisperModel is using WhisperSdpaAttention, but `torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True` or `layer_head_mask` not None. F

'[["I", 0.0, 0.12], ["dont", 0.12, 0.28], ["like", 0.28, 0.4], ["to", 0.4, 0.54], ["play", 0.54, 0.74], ["with", 0.74, 0.96], ["Monster", 0.96, 1.36], ["High", 1.36, 1.62], ["dolls.", 1.62, 2.44], ["And", 2.44, 2.98], ["I", 2.98, 3.12], ["will", 3.12, 3.26], ["give", 3.26, 3.46], ["half", 3.46, 3.74], ["of", 3.74, 3.9], ["the", 3.9, 4.02], ["Ore", 4.02, 4.2], ["I", 4.2, 4.4], ["get", 4.4, 4.5], ["in", 4.5, 4.64], ["Valheim", 4.64, 5.04], ["to", 5.04, 5.24], ["my", 5.24, 5.38], ["best", 5.38, 5.66], ["brother", 5.66, 5.92], ["in", 5.92, 6.08], ["the", 6.08, 6.16], ["world.", 6.16, 6.64]]'

In [5]:
text = "Once upon a time, there was a magical forest full of wondrous sights and sounds. The trees were tall and mighty, their leaves rustling in the gentle breeze. Colorful flowers bloomed everywhere, and the birds sang sweet melodies, welcoming all who entered. In this enchanting forest lived a little fox named Fanny. Fanny was unlike any other fox. She was curious and loved to explore every nook and cranny of her home. One bright sunny morning, while Fanny was playfully darting between the trees, she heard a soft chirping sound that caught her attention. 'What could that be?' she wondered. Without a second thought, Fanny followed the sound, eager to see who needed help."
for i in range(10):
    voice_pipe(text, "en")

/home/pavlo/miniconda3/lib/python3.12/site-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


Execution time of voice_pipe: 10.480762243270874 seconds


/home/pavlo/miniconda3/lib/python3.12/site-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


Execution time of voice_pipe: 10.159719944000244 seconds


/home/pavlo/miniconda3/lib/python3.12/site-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


...

Execution time of voice_pipe: 10.45304012298584 seconds


/home/pavlo/miniconda3/lib/python3.12/site-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


Execution time of voice_pipe: 10.364243268966675 seconds


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
/home/pavlo/miniconda3/lib/python3.12/site-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


Execution time of voice_pipe: 10.336084842681885 seconds
